# Day 8 Exercises: Data Analysis with pandas and matplotlib

Answer each question with code. Run the setup cell first!
Solutions are in `solutions/solutions.ipynb`. Try it yourself first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA = "../../data/"   # the day's data/ folder
penguins = pd.read_csv(DATA + "penguins.csv")
tickers = ["AAPL", "MSFT", "NVDA", "KO", "SPY"]
closes = pd.DataFrame({t: pd.read_csv(DATA + f"{t}.csv", parse_dates=["Date"], index_col="Date")["Close"] for t in tickers})
spy = pd.read_csv(DATA + "SPY.csv", parse_dates=["Date"], index_col="Date")
aapl = pd.read_csv(DATA + "AAPL.csv", parse_dates=["Date"], index_col="Date")
print("Loaded!", penguins.shape, closes.shape)

## Part A: Penguins 🐧

**A1.** How many penguins were measured on each island?

In [ ]:
penguins["island"].value_counts()

**A2.** What's the average bill length of each species? Sort from longest to shortest.

In [ ]:
penguins.groupby("species")["bill_length_mm"].mean().sort_values(ascending=False).round(1)

**A3.** Find the single heaviest penguin. What species, island, and sex is it?

In [ ]:
penguins.loc[penguins["body_mass_g"].idxmax(), ["species", "island", "sex", "body_mass_g"]]

**A4.** For each species, what *fraction* of the penguins (with known sex) are female? *(Hint: the mean of a True/False column is the fraction that's True.)*

In [ ]:
known = penguins.dropna(subset=["sex"])
(known["sex"] == "FEMALE").groupby(known["species"]).mean().round(3)

**A5.** How many Chinstrap penguins have a bill longer than 50 mm **and** weigh under 4,000 g?

In [ ]:
mask = (penguins["species"] == "Chinstrap") & (penguins["bill_length_mm"] > 50) & (penguins["body_mass_g"] < 4000)
mask.sum()

**A6.** Add a column `mass_kg` (body mass in kilograms). Then show the average `mass_kg` for every **species + sex** combination.

In [ ]:
penguins["mass_kg"] = penguins["body_mass_g"] / 1000
penguins.groupby(["species", "sex"])["mass_kg"].mean().round(2)

**A7.** Plot overlapping histograms of body mass, one color per species, with a legend and axis labels.

In [ ]:
for species in penguins["species"].unique():
    plt.hist(penguins[penguins["species"] == species]["body_mass_g"].dropna(), bins=15, alpha=0.6, label=species)
plt.xlabel("Body mass (g)")
plt.ylabel("Count")
plt.title("Body mass by species")
plt.legend()
plt.show()

**A8.** Make a scatter plot of flipper length (x) vs body mass (y), colored by **island**.

In [ ]:
for island in penguins["island"].unique():
    g = penguins[penguins["island"] == island]
    plt.scatter(g["flipper_length_mm"], g["body_mass_g"], label=island, alpha=0.6)
plt.xlabel("Flipper length (mm)")
plt.ylabel("Body mass (g)")
plt.legend()
plt.show()

## Part B: Stocks 📈

**B1.** On what date did SPY have its highest closing price, and what was it?

In [ ]:
print(spy["Close"].idxmax().date(), spy["Close"].max())

**B2.** Compute SPY's return for each calendar year: (last close of the year / first close of the year) − 1. *(Hint: `spy['Close'].groupby(spy.index.year)`, then `.first()` and `.last()`. Note that 2021 and 2026 are only partial years in this data.)*

In [ ]:
yearly = spy["Close"].groupby(spy.index.year)
(yearly.last() / yearly.first() - 1).round(3)

**B3.** On how many days did SPY move more than 2% (up **or** down)? *(Hint: `.abs()`.)*

In [ ]:
spy_returns = spy["Close"].pct_change()
(spy_returns.abs() > 0.02).sum()

**B4.** Plot AAPL and NVDA for **2024 only**, both normalized to start at 100.

In [ ]:
both = closes.loc["2024", ["AAPL", "NVDA"]]
(both / both.iloc[0] * 100).plot(figsize=(10, 4), title="2024: AAPL vs NVDA (start = 100)")
plt.ylabel("Value")
plt.show()

**B5.** ⭐ **Max drawdown** is the worst drop from a previous peak. For each ticker, compute `(price / price.cummax() - 1).min()`. Which one had the scariest drop?

In [ ]:
drawdowns = (closes / closes.cummax() - 1).min().sort_values()
drawdowns.round(3)

**B6.** ⭐ Make a bar chart of the average **volume** of AAPL for each day of the week. Is any day busier? *(Hint: `aapl.index.day_name()`.)*

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
by_day = aapl["Volume"].groupby(aapl.index.day_name()).mean().reindex(order)
plt.bar(by_day.index, by_day.values / 1e6)
plt.ylabel("Average volume (millions of shares)")
plt.show()